## 무신사 리뷰 데이터 수집

### 휠라 리뷰 데이터
##### 의류 카테고리의 경우 표본이 5건 이하여서 수집 제외, 브랜드 주력 카테고리인 신발 위주 리뷰 수집 진행

In [5]:
import pandas as pd
from curl_cffi import requests
import time
import os
import json
import logging
from datetime import datetime

# ==========================================
# 로깅 설정
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

CHECKPOINT_FILE = "checkpoint.txt"
OUTPUT_FILE = "musinsa_fila_shoes_reviews.csv"


# ==========================================
# 체크포인트
# ==========================================
def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return set()
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return {int(l.strip()) for l in f if l.strip().isdigit()}

def save_checkpoint(goods_no):
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(f"{goods_no}\n")


# ==========================================
# API 진단 (최초 1회)
# ==========================================
def diagnose_api(goods_no):
    log.info(f"🔍 API 진단 시작 (상품 {goods_no} 기준)...")

    candidates = [
        {"name": "타입별_page1", "start_page": 1, "use_type": True},
        {"name": "타입별_page0", "start_page": 0, "use_type": True},
        {"name": "전체_page1",   "start_page": 1, "use_type": False},
        {"name": "전체_page0",   "start_page": 0, "use_type": False},
    ]

    BASE_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
    session  = requests.Session(impersonate="chrome124")
    headers  = {
        "accept":          "application/json, text/plain, */*",
        "accept-language": "ko-KR,ko;q=0.9",
        "referer":         f"https://www.musinsa.com/products/{goods_no}",
        "user-agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }

    for cand in candidates:
        params_str = "&typeList=style" if cand["use_type"] else ""
        url = (
            f"{BASE_URL}?page={cand['start_page']}&pageSize=20"
            f"&goodsNo={goods_no}&sort=new{params_str}"
        )
        try:
            res = session.get(url, headers=headers, timeout=10)
            if res.status_code == 200:
                items = res.json().get("data", {}).get("list", [])
                log.info(f"  ✅ 성공: [{cand['name']}] — 리뷰 {len(items)}건 확인")
                cand["base_url"] = BASE_URL
                return cand
        except Exception:
            pass
        time.sleep(1)

    log.error("  ❌ 모든 API 패턴 진단 실패")
    return None


# ==========================================
# 설문 파싱 헬퍼
# ==========================================
def parse_survey(item):
    """reviewSurveySatisfaction 에서 속성별 답변 추출"""
    result = {
        "survey_size":    None,
        "survey_width":   None,
        "survey_comfort": None,
        "survey_instep":  None,
    }

    attr_map = {
        "사이즈":     "survey_size",
        "발볼 넓이":  "survey_width",
        "착용감":     "survey_comfort",
        "발등 압박감": "survey_instep",
    }

    survey = item.get("reviewSurveySatisfaction") or {}
    for q in survey.get("questions") or []:
        attr    = q.get("attribute", "")
        answers = q.get("answers") or []
        key     = attr_map.get(attr)
        if key and answers:
            result[key] = answers[0].get("answerShortText")

    return result


# ==========================================
# 리뷰 수집
# ==========================================
def get_musinsa_reviews(goods_no, api_config, page_limit=175, max_retries=3):
    all_data   = []
    base_url   = api_config["base_url"]
    start_page = api_config["start_page"]
    collect_dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if api_config["use_type"]:
        review_types = [
            {"name": "스타일",          "params": "typeList=style"},
            {"name": "일반_포토_체험단", "params": "typeList=general&typeList=photo&typeList=goods&typeList=experience"},
            {"name": "한달후기",         "params": "typeList=month"},
        ]
    else:
        review_types = [{"name": "전체", "params": ""}]

    session = requests.Session(impersonate="chrome124")
    headers = {
        "accept":          "application/json, text/plain, */*",
        "accept-language": "ko-KR,ko;q=0.9",
        "referer":         f"https://www.musinsa.com/products/{goods_no}",
        "user-agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }

    for r_type in review_types:
        params_str = f"&{r_type['params']}" if r_type["params"] else ""

        for page in range(start_page, start_page + page_limit):
            page_success = False

            for attempt in range(1, max_retries + 1):
                try:
                    url = (
                        f"{base_url}?page={page}&pageSize=20"
                        f"&goodsNo={goods_no}&sort=new{params_str}"
                    )
                    res = session.get(url, headers=headers, timeout=15)

                    if res.status_code == 400:
                        log.warning(f"  400: {goods_no}/{r_type['name']}/p{page} — {res.text[:200]}")
                        break

                    if res.status_code == 403:
                        log.error(f"  403 차단: {goods_no} — 60초 대기")
                        time.sleep(60)
                        continue

                    if res.status_code == 429:
                        wait = 30 * attempt
                        log.warning(f"  429 제한: {wait}초 대기")
                        time.sleep(wait)
                        continue

                    if res.status_code != 200:
                        log.warning(f"  HTTP {res.status_code}: {goods_no}/{r_type['name']}/p{page}")
                        break

                    try:
                        data = res.json()
                    except json.JSONDecodeError as e:
                        log.error(f"  JSON 파싱 실패: {e} | {res.text[:200]}")
                        break

                    payload = data.get("data") or {}
                    items   = (
                        payload.get("list")
                        or payload.get("reviews")
                        or payload.get("content")
                        or []
                    )

                    if not items:
                        log.info(f"  {goods_no}/{r_type['name']}/p{page} — 빈 페이지, 타입 종료")
                        break

                    for item in items:
                        goods        = item.get("goods") or {}
                        profile      = item.get("userProfileInfo") or {}
                        images       = item.get("images") or []
                        survey       = parse_survey(item)
                        review_no    = item.get("no")
                        image_urls   = "|".join(
                            f"https://www.musinsa.com{img['imageUrl']}"
                            for img in images if img.get("imageUrl")
                        )

                        all_data.append({
                            # ── 기본 수집 정보 ──
                            "collect_date":   collect_dt,
                            "review_date":    (item.get("createDate") or "")[:10],

                            # ── 상품 정보 ──
                            "product_id":     str(goods.get("goodsNo", goods_no)),
                            "product_name":   goods.get("goodsName", ""),

                            # ── 리뷰 핵심 ──
                            "review_no":      review_no,
                            "review_type":    item.get("typeName", r_type["name"]),
                            "rating":         int(item.get("grade") or 0),
                            "review_text":    (item.get("content") or "").replace("\n", " ").strip(),

                            # ── 작성자 ──
                            "author":         profile.get("userNickName", ""),
                            "user_height":    profile.get("userHeight") or None,
                            "user_weight":    profile.get("userWeight") or None,

                            # ── 옵션 / 반응 ──
                            "option":         item.get("goodsOption", ""),
                            "helpful":        int(item.get("likeCount") or 0),
                            "comment_count":  int(item.get("commentCount") or 0),

                            # ── 이미지 ──
                            "has_image":      len(images) > 0,
                            "image_urls":     image_urls,

                            # ── 설문 (신발 핏 정보) ──
                            "survey_size":    survey["survey_size"],
                            "survey_width":   survey["survey_width"],
                            "survey_comfort": survey["survey_comfort"],
                            "survey_instep":  survey["survey_instep"],

                            # ── URL ──
                            "url": f"https://www.musinsa.com/products/{goods.get('goodsNo', goods_no)}#review_{review_no}",
                        })

                    log.info(f"  ✔ {goods_no}/{r_type['name']}/p{page} — {len(items)}건")
                    page_success = True
                    time.sleep(0.8)
                    break

                except requests.exceptions.Timeout:
                    log.warning(f"  타임아웃: {goods_no}/p{page}/attempt{attempt}")
                    time.sleep(5 * attempt)
                except Exception as e:
                    log.error(f"  예외 {type(e).__name__}: {e}")
                    break

            if not page_success:
                break

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    return df.drop_duplicates(subset=["review_no"]).reset_index(drop=True)


# ==========================================
# CSV 저장
# ==========================================
def save_to_csv(df: pd.DataFrame, path: str):
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False, encoding="utf-8-sig")


# ==========================================
# 메인 실행
# ==========================================
if __name__ == "__main__":

    target_list = [
        3892794, 4328804, 4469887, 4699900, 4791166, 4922473, 4990088, 3892785, 4922467, 5311939,
        5908978, 5475849, 5312026, 5338881, 5284696, 5825666, 5312098, 5908988, 5312080, 5985221,
        5284659, 4663147, 4789032, 5825668, 5311984, 4952235, 6010742, 5825671, 5267322, 5908996,
        5056743, 6063343, 5056750, 5056745, 5341288, 5267329, 4789036, 6010743, 5056747, 4952237,
        4789038, 6023755, 6022848, 6022846, 6022681, 6010936, 6010746, 6006000, 5870088, 5791439,
        5611332, 5506136, 5488286, 5433878, 6190665, 6023760, 6010933, 6009591, 6009587, 6009532,
        6009475, 6006027, 6006014, 5985225, 5892873, 5870089, 5870064, 5858596, 5858591, 5858590,
        5858589, 5825670, 5693788, 5488290, 5488289, 5488276, 5488266, 5256885, 5232295, 4952242,
        4780187, 6289935, 6289884, 6289803, 6190666, 6190660, 6190658, 6190651, 6190645, 6190642,
        6190574, 6190572, 6190563, 6137256, 6137251, 6137250, 6136819, 6136818, 6136816, 6136813,
        6082140, 6082134, 6082131, 6079640, 6079628, 6079624, 6079620, 6079589, 6079584, 6023751,
        6022841, 6022838, 6022836, 6022832, 6022829, 6009578, 6009573, 6009570, 6009565, 6009525,
        6009522, 6009518, 6009517, 6009513, 6009510, 6009505, 6009503, 6009502, 6009501, 6009490,
        6009489, 6009486, 6009483, 6009481, 5892580, 5892576, 5870090, 5870085, 5870083, 5870081,
        5870080, 5870075, 5870072, 5870068, 5858604, 5858602, 5858599, 5858594, 5825676, 5791440,
        5766782, 5766778, 5693787, 5693784, 5611338, 5611336, 5506133, 5488269, 5488261, 5488255,
        5433880, 5267311, 5232291, 5232288, 5120384, 4952239, 4789040, 4789034, 4720283,
    ]

    done      = load_checkpoint()
    remaining = [g for g in target_list if g not in done]

    if not remaining:
        log.info("✅ 모든 상품이 이미 완료되었습니다.")
    else:
        api_config = diagnose_api(remaining[0])
        if not api_config:
            raise SystemExit("❌ API 진단 실패 — 네트워크 또는 차단 여부를 확인하세요.")

        log.info(f"🚀 수집 시작 — 설정: [{api_config['name']}] / {len(remaining)}개 남음")

        for idx, g_no in enumerate(remaining, 1):
            log.info(f"▶ [{idx}/{len(remaining)}] 상품 {g_no}")

            df = get_musinsa_reviews(g_no, api_config, page_limit=175)

            if not df.empty:
                save_to_csv(df, OUTPUT_FILE)
                log.info(f"  💾 {len(df)}건 저장 완료")
            else:
                log.info(f"  📭 리뷰 없음")

            save_checkpoint(g_no)
            time.sleep(3.5)

        log.info(f"🎉 전체 완료! 저장 파일: {OUTPUT_FILE}")

2026-04-21 17:04:23,707 [INFO] 🔍 API 진단 시작 (상품 3892794 기준)...
2026-04-21 17:04:23,826 [INFO]   ✅ 성공: [타입별_page1] — 리뷰 20건 확인
2026-04-21 17:04:23,827 [INFO] 🚀 수집 시작 — 설정: [타입별_page1] / 169개 남음
2026-04-21 17:04:23,827 [INFO] ▶ [1/169] 상품 3892794
2026-04-21 17:04:23,935 [INFO]   ✔ 3892794/스타일/p1 — 20건
2026-04-21 17:04:24,826 [INFO]   ✔ 3892794/스타일/p2 — 20건
2026-04-21 17:04:25,779 [INFO]   ✔ 3892794/스타일/p3 — 20건
2026-04-21 17:04:26,657 [INFO]   ✔ 3892794/스타일/p4 — 20건
2026-04-21 17:04:27,557 [INFO]   ✔ 3892794/스타일/p5 — 20건
2026-04-21 17:04:28,466 [INFO]   ✔ 3892794/스타일/p6 — 20건
2026-04-21 17:04:29,338 [INFO]   ✔ 3892794/스타일/p7 — 20건
2026-04-21 17:04:30,196 [INFO]   ✔ 3892794/스타일/p8 — 20건
2026-04-21 17:04:31,087 [INFO]   ✔ 3892794/스타일/p9 — 20건
2026-04-21 17:04:31,948 [INFO]   ✔ 3892794/스타일/p10 — 20건
2026-04-21 17:04:32,826 [INFO]   ✔ 3892794/스타일/p11 — 20건
2026-04-21 17:04:33,696 [INFO]   ✔ 3892794/스타일/p12 — 20건
2026-04-21 17:04:34,557 [INFO]   ✔ 3892794/스타일/p13 — 20건
2026-04-21 17:04:35,416 

### 룰루레몬 리뷰 데이터 수집

In [1]:
import pandas as pd
from curl_cffi import requests
import time
import os
import json
import logging
from datetime import datetime

# ==========================================
# 로깅 설정
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

CHECKPOINT_FILE = "checkpoint.txt"
OUTPUT_FILE = "musinsa_lululemon_reviews.csv"


# ==========================================
# 체크포인트
# ==========================================
def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return set()
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return {int(l.strip()) for l in f if l.strip().isdigit()}

def save_checkpoint(goods_no):
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(f"{goods_no}\n")


# ==========================================
# API 진단 (최초 1회)
# ==========================================
def diagnose_api(goods_no):
    log.info(f"🔍 API 진단 시작 (상품 {goods_no} 기준)...")

    # ✅ 핵심 수정: page0 후보를 page1보다 먼저 배치
    candidates = [
        {"name": "타입별_page0", "start_page": 0, "use_type": True},
        {"name": "전체_page0",   "start_page": 0, "use_type": False},
        {"name": "타입별_page1", "start_page": 1, "use_type": True},
        {"name": "전체_page1",   "start_page": 1, "use_type": False},
    ]

    BASE_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
    session  = requests.Session(impersonate="chrome124")
    headers  = {
        "accept":          "application/json, text/plain, */*",
        "accept-language": "ko-KR,ko;q=0.9",
        "referer":         f"https://www.musinsa.com/products/{goods_no}",
        "user-agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }

    for cand in candidates:
        params_str = "&typeList=style" if cand["use_type"] else ""
        url = (
            f"{BASE_URL}?page={cand['start_page']}&pageSize=20"
            f"&goodsNo={goods_no}&sort=new{params_str}"
        )
        try:
            res = session.get(url, headers=headers, timeout=10)
            if res.status_code == 200:
                data    = res.json()
                payload = data.get("data", {})
                items   = (
                    payload.get("list")
                    or payload.get("reviews")
                    or payload.get("content")
                    or []
                )
                total = (
                    payload.get("totalCount")
                    or payload.get("total")
                    or payload.get("totalElements")
                )
                total_str = f" / 총 {total}건" if total is not None else ""
                log.info(f"  ✅ 성공: [{cand['name']}] — 리뷰 {len(items)}건 확인{total_str}")
                cand["base_url"] = BASE_URL
                return cand
        except Exception:
            pass
        time.sleep(1)

    log.error("  ❌ 모든 API 패턴 진단 실패")
    return None


# ==========================================
# 설문 파싱 헬퍼
# ==========================================
def parse_survey(item):
    result = {
        "survey_size":    None,
        "survey_width":   None,
        "survey_comfort": None,
        "survey_instep":  None,
    }
    attr_map = {
        "사이즈":      "survey_size",
        "발볼 넓이":  "survey_width",
        "착용감":      "survey_comfort",
        "발등 압박감": "survey_instep",
    }
    survey = item.get("reviewSurveySatisfaction") or {}
    for q in survey.get("questions") or []:
        attr    = q.get("attribute", "")
        answers = q.get("answers") or []
        key     = attr_map.get(attr)
        if key and answers:
            result[key] = answers[0].get("answerShortText")
    return result


# ==========================================
# 리뷰 수집
# ==========================================
def get_musinsa_reviews(goods_no, api_config, page_limit=175, max_retries=3):
    all_data   = []
    base_url   = api_config["base_url"]
    start_page = api_config["start_page"]
    collect_dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if api_config["use_type"]:
        review_types = [
            {"name": "스타일",          "params": "typeList=style"},
            {"name": "일반_포토_체험단", "params": "typeList=general&typeList=photo&typeList=goods&typeList=experience"},
            {"name": "한달후기",         "params": "typeList=month"},
        ]
    else:
        review_types = [{"name": "전체", "params": ""}]

    session = requests.Session(impersonate="chrome124")
    headers = {
        "accept":          "application/json, text/plain, */*",
        "accept-language": "ko-KR,ko;q=0.9",
        "referer":         f"https://www.musinsa.com/products/{goods_no}",
        "user-agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }

    for r_type in review_types:
        params_str   = f"&{r_type['params']}" if r_type["params"] else ""
        type_total   = None
        type_fetched = 0

        for page in range(start_page, start_page + page_limit):
            page_success = False

            for attempt in range(1, max_retries + 1):
                try:
                    url = (
                        f"{base_url}?page={page}&pageSize=20"
                        f"&goodsNo={goods_no}&sort=new{params_str}"
                    )
                    res = session.get(url, headers=headers, timeout=15)

                    if res.status_code == 400:
                        log.warning(f"  400: {goods_no}/{r_type['name']}/p{page} — {res.text[:200]}")
                        break

                    if res.status_code == 403:
                        log.error(f"  403 차단: {goods_no} — 60초 대기")
                        time.sleep(60)
                        continue

                    if res.status_code == 429:
                        wait = 30 * attempt
                        log.warning(f"  429 제한: {wait}초 대기")
                        time.sleep(wait)
                        continue

                    if res.status_code != 200:
                        log.warning(f"  HTTP {res.status_code}: {goods_no}/{r_type['name']}/p{page}")
                        break

                    try:
                        data = res.json()
                    except json.JSONDecodeError as e:
                        log.error(f"  JSON 파싱 실패: {e} | {res.text[:200]}")
                        break

                    payload = data.get("data") or {}

                    if type_total is None:
                        type_total = (
                            payload.get("totalCount")
                            or payload.get("total")
                            or payload.get("totalElements")
                        )
                        if type_total is not None:
                            log.info(f"  📊 {goods_no}/{r_type['name']} — API 총 건수: {type_total}")

                    # ✅ 빈 리스트와 None을 구분하여 정확히 판단
                    raw_list = (
                        payload.get("list")
                        if payload.get("list") is not None
                        else payload.get("reviews")
                        if payload.get("reviews") is not None
                        else payload.get("content")
                        if payload.get("content") is not None
                        else []
                    )

                    if not raw_list:
                        log.info(f"  {goods_no}/{r_type['name']}/p{page} — 빈 페이지, 타입 종료")
                        break

                    for item in raw_list:
                        goods        = item.get("goods") or {}
                        profile      = item.get("userProfileInfo") or {}
                        images       = item.get("images") or []
                        survey       = parse_survey(item)
                        review_no    = item.get("no")
                        image_urls   = "|".join(
                            f"https://www.musinsa.com{img['imageUrl']}"
                            for img in images if img.get("imageUrl")
                        )

                        all_data.append({
                            "collect_date":   collect_dt,
                            "review_date":    (item.get("createDate") or "")[:10],
                            "product_id":     str(goods.get("goodsNo", goods_no)),
                            "product_name":   goods.get("goodsName", ""),
                            "review_no":      review_no,
                            "review_type":    item.get("typeName", r_type["name"]),
                            "rating":         int(item.get("grade") or 0),
                            "review_text":    (item.get("content") or "").replace("\n", " ").strip(),
                            "author":         profile.get("userNickName", ""),
                            "user_height":    profile.get("userHeight") or None,
                            "user_weight":    profile.get("userWeight") or None,
                            "option":         item.get("goodsOption", ""),
                            "helpful":        int(item.get("likeCount") or 0),
                            "comment_count":  int(item.get("commentCount") or 0),
                            "has_image":      len(images) > 0,
                            "image_urls":     image_urls,
                            "survey_size":    survey["survey_size"],
                            "survey_width":   survey["survey_width"],
                            "survey_comfort": survey["survey_comfort"],
                            "survey_instep":  survey["survey_instep"],
                            "url": f"https://www.musinsa.com/products/{goods.get('goodsNo', goods_no)}#review_{review_no}",
                        })

                    type_fetched += len(raw_list)
                    log.info(f"  ✔ {goods_no}/{r_type['name']}/p{page} — {len(raw_list)}건 (누적: {type_fetched})")
                    page_success = True
                    time.sleep(0.8)
                    break

                except requests.exceptions.Timeout:
                    log.warning(f"  타임아웃: {goods_no}/p{page}/attempt{attempt}")
                    time.sleep(5 * attempt)
                except Exception as e:
                    log.error(f"  예외 {type(e).__name__}: {e}")
                    break

            if not page_success:
                break

        if type_total is not None and type_fetched < type_total:
            log.warning(
                f"  ⚠️ {goods_no}/{r_type['name']} — "
                f"수집 {type_fetched}건 / API 총 {type_total}건 (미수집 {type_total - type_fetched}건)"
            )

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    return df.drop_duplicates(subset=["review_no"]).reset_index(drop=True)


# ==========================================
# CSV 저장
# ==========================================
def save_to_csv(df: pd.DataFrame, path: str):
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False, encoding="utf-8-sig")


# ==========================================
# 메인 실행
# ==========================================
target_list = [
    5733085, 5785396, 5737750, 5748893, 5749770, 5732869, 5732069, 5732560, 5776137, 5785397,
    5738183, 5744103, 5737935, 5776189, 5749765, 5737813, 5850230, 5785362, 5738207, 5732356,
    5859709, 5798440, 5748989, 5732789, 5732678, 6027345, 5749784, 5737934, 5737804, 5732140,
    5798381, 5776196, 5738485, 5900254, 5798433, 5744032, 5738097, 5737698, 5733076, 5732633,
    5912196, 5802000, 5785324, 5749384, 5738380, 5737689, 5732956, 5732950, 6181058, 6137724,
    6079094, 6045102, 6027347, 5879401, 5829822, 5738471, 5737662, 5733207, 5732886, 5732677,
    5732531, 5732299, 6180961, 6142564, 6105838, 5965342, 5879262, 5879239, 5831127, 5798388,
    5798240, 5785323, 5749868, 5748984, 5738556, 5737890, 5733199, 5733140, 5733029, 5732827,
    5732790, 5732777, 5732612, 5732472, 6137659, 6070017, 6069948, 5965321, 5907542, 5906606,
    5886290, 5859703, 5829820, 5813653, 5798450, 5798384, 5785961, 5776173, 5749835, 5749834,
    5749025, 5748988, 5748881, 5738563, 5737712, 5733208, 5732960, 5732922, 5732767, 5732249,
    5731867, 6167511, 6142571, 6142554, 6142208, 6142190, 6137645, 6106034, 6106030, 6105863,
    6079152, 6079149, 5965345, 5965320, 5965311, 5925372, 5895633, 5879279, 5879244, 5831104,
    5813655, 5813654, 5813238, 5809527, 5798447, 5798412, 5798395, 5798365, 5798279, 5797883,
    5785364, 5781076, 5781062, 5780070, 5776114, 5776092, 5775974, 5749772, 5748884, 5738526,
    5738507, 5738364, 5738204, 5738123, 5738119, 5737719, 5733141, 5733089, 5733083, 5733056,
    5733028, 5732990, 5732923, 5732722, 5732516, 5732366, 5732259, 5732106, 5731584, 6232351,
    6232344, 6192450, 6192431, 6191702, 6180973, 6180912, 6180746, 6167522, 6167352, 6158181,
    6142212, 6142195, 6137744, 6137733, 6106369, 6106354, 6106120, 6106103, 6105925, 6105921,
    6105852, 6105834, 6105795, 6079067, 6069960, 6069944, 6069941, 6051215, 6022223, 6022221,
    5976908, 5971279, 5965323, 5925375, 5912190, 5906712, 5906306, 5901055, 5891868, 5886348,
    5886292, 5879348, 5879340, 5879335, 5879254, 5879252, 5876092, 5869716, 5850229, 5834186,
    5831140, 5831139, 5829696, 5813250, 5802004, 5798555, 5798468, 5798404, 5798393, 5798253,
    5785924, 5785381, 5780103, 5780076, 5776166, 5776139, 5776132, 5776107, 5776070, 5749858,
    5748895, 5738498, 5738342, 5738322, 5738188, 5738151, 5738133, 5738122, 5737975, 5737802,
    5737720, 5737702, 5733176, 5733162, 5733126, 5733054, 5733002, 5732975, 5732958, 5732820,
    5732793, 5732775, 5732740, 5732729, 5732711, 5732701, 5732683, 5732635, 5732559, 5732549,
    5732535, 5732492, 5732490, 5732474, 5732369, 5732359, 5732335, 5732323, 5732316, 5732247,
    5732064, 5732049, 5732012, 5731786, 5731564, 6321827, 6232446, 6232393, 6192458, 6192449,
    6191704, 6181288, 6181080, 6181060, 6180989, 6180920, 6180864, 6180843, 6180816, 6180786,
    6180776, 6180761, 6180740, 6180726, 6167528, 6167514, 6167479, 6167357, 6142572, 6142551,
    6142176, 6137741, 6137736, 6137698, 6137642, 6106486, 6106387, 6106146, 6106118, 6106095,
    6106085, 6106080, 6106079, 6106048, 6106039, 6106008, 6105997, 6105993, 6105992, 6105933,
    6105910, 6105881, 6105874, 6105850, 6105836, 6105830, 6105818, 6105811, 6105797, 6105781,
    6079188, 6079086, 6079083, 6079079, 6079057, 6070038, 6069954, 6045095, 6028298, 5971274,
    5965340, 5965337, 5945793, 5938349, 5938340, 5925387, 5912200, 5912195, 5907385, 5906669,
    5906494, 5906073, 5905940, 5900483, 5900257, 5900256, 5891852, 5891839, 5891814, 5886277,
    5886275, 5879389, 5879373, 5879371, 5879318, 5879315, 5879301, 5879273, 5879251, 5859701,
    5850132, 5850131, 5850100, 5837859, 5837856, 5834642, 5834169, 5834150, 5831153, 5831150,
    5831136, 5831135, 5831131, 5831129, 5831122, 5831073, 5830881, 5829761, 5827977, 5824171,
    5817185, 5816665, 5813656, 5813357, 5813264, 5802001, 5798458, 5798402, 5798335, 5798333,
    5798294, 5798255, 5798254, 5797869, 5785319, 5781068, 5780085, 5776171, 5776168, 5776138,
    5776090, 5776071, 5776068, 5776066, 5775741, 5774955, 5749777, 5749771, 5749740, 5748990,
    5748935, 5744125, 5744069, 5744024, 5738605, 5738581, 5738555, 5738408, 5738390, 5738336,
    5738303, 5738130, 5738126, 5738098, 5738033, 5737973, 5737958, 5737944, 5737840, 5737838,
    5737748, 5737668, 5733200, 5733183, 5733181, 5733172, 5733160, 5733119, 5733118, 5733097,
    5733093, 5733088, 5733067, 5733062, 5733001, 5732985, 5732973, 5732932, 5732926, 5732894,
    5732794, 5732769, 5732766, 5732755, 5732718, 5732663, 5732628, 5732592, 5732476, 5732320,
    5732264, 5732245, 5732221, 5731952, 5731865
]

done      = load_checkpoint()
remaining = [g for g in target_list if g not in done]

if not remaining:
    log.info("✅ 모든 상품이 이미 완료되었습니다.")
else:
    api_config = diagnose_api(remaining[0])
    if not api_config:
        raise SystemExit("❌ API 진단 실패 — 네트워크 또는 차단 여부를 확인하세요.")

    log.info(f"🚀 수집 시작 — 설정: [{api_config['name']}] / {len(remaining)}개 남음")

    for idx, g_no in enumerate(remaining, 1):
        log.info(f"▶ [{idx}/{len(remaining)}] 상품 {g_no}")

        df = get_musinsa_reviews(g_no, api_config, page_limit=175)

        if not df.empty:
            save_to_csv(df, OUTPUT_FILE)
            log.info(f"  💾 {len(df)}건 저장 완료")
        else:
            log.info(f"  📭 리뷰 없음")

        save_checkpoint(g_no)
        time.sleep(3.5)

    log.info(f"🎉 전체 완료! 저장 파일: {OUTPUT_FILE}")

2026-04-21 22:56:54,754 [INFO] ✅ 모든 상품이 이미 완료되었습니다.


### 안다르 스포츠/레저 리뷰 데이터 수집

In [3]:
# ============================================================
# 무신사 안다르 리뷰 크롤러 — 단일 셀 실행 버전
# ============================================================
import pandas as pd
from curl_cffi import requests
import time
import os
import json
import logging
from datetime import datetime
from tqdm.auto import tqdm

# ── 파일 경로 ──
OUTPUT_FILE     = "musinsa_andar_reviews.csv"
CHECKPOINT_FILE = "checkpoint.json"
LOG_FILE        = "scraper.log"

# ── 크롤링 파라미터 ──
PAGE_SIZE          = 20
PAGE_LIMIT         = 80
SLEEP_BETWEEN_PAGE = 0.8
SLEEP_BETWEEN_PROD = 3.0
MAX_RETRIES        = 3
BASE_URL = "https://goods.musinsa.com/api2/review/v1/view/list"

# ── 로깅 (Jupyter 중복 핸들러 방지) ──
log = logging.getLogger("musinsa")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
_fh = logging.FileHandler(LOG_FILE, encoding="utf-8"); _fh.setFormatter(_fmt)
_sh = logging.StreamHandler(); _sh.setFormatter(_fmt)
log.addHandler(_fh); log.addHandler(_sh)
log.propagate = False

# ============================================================
# 체크포인트
# ============================================================
def load_checkpoint() -> dict:
    if not os.path.exists(CHECKPOINT_FILE):
        return {}
    try:
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        log.warning("체크포인트 파일 손상 — 새로 시작합니다.")
        return {}

def save_checkpoint(state: dict):
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)

# ============================================================
# 세션 / 헤더
# ============================================================
def build_session():
    return requests.Session(impersonate="chrome124")

def build_headers(goods_no):
    return {
        "accept": "application/json, text/plain, */*",
        "accept-language": "ko-KR,ko;q=0.9",
        "referer": f"https://www.musinsa.com/products/{goods_no}",
        "user-agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
    }

# ============================================================
# API 진단
# ============================================================
def diagnose_api(session, goods_no):
    log.info(f"🔍 API 진단 (상품 {goods_no})")
    candidates = [
        {"name": "type-style/p0", "start_page": 0, "use_type": True},
        {"name": "type-style/p1", "start_page": 1, "use_type": True},
        {"name": "전체/p0",        "start_page": 0, "use_type": False},
        {"name": "전체/p1",        "start_page": 1, "use_type": False},
    ]
    for c in candidates:
        qs = "&typeList=style" if c["use_type"] else ""
        url = (
            f"{BASE_URL}?page={c['start_page']}&pageSize={PAGE_SIZE}"
            f"&goodsNo={goods_no}&selectedSimilarNo={goods_no}&sort=new{qs}"
        )
        try:
            r = session.get(url, headers=build_headers(goods_no), timeout=10)
            if r.status_code == 200:
                payload = r.json().get("data", {})
                items   = payload.get("list") or []

                # ✅ 버그 수정: total을 숫자로 안전하게 변환
                raw_total = payload.get("total")
                try:
                    total = int(raw_total) if raw_total is not None else 0
                except (ValueError, TypeError):
                    total = 0

                # page0 에서 아이템이 없고 총 리뷰가 존재하면 → 1-indexed API 가능성
                # → page1 후보로 넘어감
                if c["start_page"] == 0 and not items and total > 0:
                    log.debug(f"  [{c['name']}] page0 빈 응답, total={total} → 다음 후보 시도")
                    continue

                log.info(f"  ✅ [{c['name']}] 총 리뷰 수: {total}건 감지")
                return c
        except Exception as e:
            log.debug(f"  진단 실패 [{c['name']}]: {e}")
        time.sleep(1)
    return None

# ============================================================
# 설문 파서 (동적) + 이미지 URL
# ============================================================
def parse_survey(item) -> dict:
    out = {}
    survey = item.get("reviewSurveySatisfaction") or {}
    for q in survey.get("questions") or []:
        attr    = (q.get("attribute") or "").strip()
        answers = q.get("answers") or []
        if attr and answers:
            out[attr] = answers[0].get("answerShortText")
    return out

def normalize_image_url(url: str) -> str:
    if not url:
        return ""
    if url.startswith("http"):
        return url
    if url.startswith("//"):
        return "https:" + url
    return "https://www.musinsa.com" + url

# ============================================================
# 페이로드에서 아이템 추출 (None / [] 구분)
# ============================================================
def extract_items(payload: dict) -> list:
    """
    ✅ 버그 수정: `or` 체인 대신 `is not None` 체크로 빈 리스트와 누락 키를 구분
    list=[] → 페이지 끝(빈 리스트 반환), list 키 없음 → 다른 키 탐색
    """
    for key in ("list", "reviews", "content"):
        val = payload.get(key)
        if val is not None:
            return val
    return []

# ============================================================
# 상품 1개 수집
# ============================================================
def collect_product(session, goods_no, api_cfg, page_limit=PAGE_LIMIT):
    collected  = []
    collect_dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    start_page = api_cfg["start_page"]

    if api_cfg["use_type"]:
        review_types = [
            {"name": "스타일",   "qs": "typeList=style"},
            {"name": "일반",     "qs": "typeList=general&typeList=photo&typeList=goods&typeList=experience"},
            {"name": "한달후기", "qs": "typeList=month"},
        ]
    else:
        review_types = [{"name": "전체", "qs": ""}]

    got_any_response = False
    had_fatal        = False

    for rt in review_types:
        qs = f"&{rt['qs']}" if rt["qs"] else ""
        consecutive_fail = 0

        for page in range(start_page, start_page + page_limit):
            url = (
                f"{BASE_URL}?page={page}&pageSize={PAGE_SIZE}"
                f"&goodsNo={goods_no}&selectedSimilarNo={goods_no}&sort=new{qs}"
            )
            page_done = False
            items     = None

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    r  = session.get(url, headers=build_headers(goods_no), timeout=15)
                    sc = r.status_code

                    if sc == 200:
                        try:
                            data = r.json()
                        except json.JSONDecodeError:
                            log.error(f"  JSON 파싱 실패 {goods_no}/{rt['name']}/p{page}")
                            break
                        payload   = data.get("data") or {}
                        items     = extract_items(payload)  # ✅ 수정된 헬퍼 사용
                        page_done = True
                        break

                    if sc == 400:
                        log.warning(f"  400 {goods_no}/{rt['name']}/p{page} — {r.text[:120]}")
                        break
                    if sc == 404:
                        log.warning(f"  404 {goods_no} — 상품 없음")
                        return pd.DataFrame(), "empty"
                    if sc == 403:
                        wait = 60 * attempt
                        log.error(f"  403 차단 — {wait}s 대기")
                        time.sleep(wait)
                        continue
                    if sc == 429:
                        wait = 30 * attempt
                        log.warning(f"  429 제한 — {wait}s 대기")
                        time.sleep(wait)
                        continue

                    log.warning(f"  HTTP {sc} {goods_no}/{rt['name']}/p{page}")
                    time.sleep(5 * attempt)

                except requests.exceptions.Timeout:
                    log.warning(f"  타임아웃 {goods_no}/p{page} attempt{attempt}")
                    time.sleep(5 * attempt)
                except Exception as e:
                    log.error(f"  예외 {type(e).__name__}: {e}")
                    time.sleep(5 * attempt)

            if not page_done:
                consecutive_fail += 1
                if consecutive_fail >= 2:
                    log.error(f"  {goods_no}/{rt['name']} — 연속 실패, 타입 중단")
                    had_fatal = True
                    break
                continue

            consecutive_fail     = 0
            got_any_response     = True

            if not items:
                log.info(f"  {goods_no}/{rt['name']}/p{page} — 빈 페이지 (타입 종료)")
                break

            for item in items:
                goods   = item.get("goods") or {}
                profile = item.get("userProfileInfo") or {}
                images  = item.get("images") or []
                survey  = parse_survey(item)
                image_urls = "|".join(
                    normalize_image_url(img.get("imageUrl", ""))
                    for img in images if img.get("imageUrl")
                )
                collected.append({
                    "collect_date":  collect_dt,
                    "review_date":   (item.get("createDate") or "")[:10],
                    "product_id":    str(goods.get("goodsNo", goods_no)),
                    "product_name":  goods.get("goodsName", ""),
                    "review_no":     item.get("no"),
                    "review_type":   item.get("typeName") or rt["name"],
                    "rating":        int(item.get("grade") or 0),
                    "review_text":   (item.get("content") or "").replace("\n", " ").strip(),
                    "author":        profile.get("userNickName", ""),
                    "user_height":   profile.get("userHeight"),
                    "user_weight":   profile.get("userWeight"),
                    "option":        item.get("goodsOption", ""),
                    "helpful":       int(item.get("likeCount") or 0),
                    "comment_count": int(item.get("commentCount") or 0),
                    "has_image":     len(images) > 0,
                    "image_urls":    image_urls,
                    "survey_json":   json.dumps(survey, ensure_ascii=False) if survey else "",
                    "url": (
                        f"https://www.musinsa.com/products/"
                        f"{goods.get('goodsNo', goods_no)}#review_{item.get('no')}"
                    ),
                })

            log.info(f"  ✔ {goods_no}/{rt['name']}/p{page} — {len(items)}건")
            time.sleep(SLEEP_BETWEEN_PAGE)

        if had_fatal:
            break

    if collected:
        df = pd.DataFrame(collected).drop_duplicates(subset=["review_no"]).reset_index(drop=True)
        return df, "ok"
    if had_fatal or not got_any_response:
        return pd.DataFrame(), "fail"
    return pd.DataFrame(), "empty"

def append_csv(df: pd.DataFrame, path: str):
    if df.empty:
        return
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False, encoding="utf-8-sig")

# ============================================================
# 타겟 리스트 (안다르)
# ============================================================
TARGET_LIST = [
    3297519, 3379413, 3414542, 3351097, 2722562, 4775728, 4535770, 4555810, 4560936, 4284335,
    4724807, 4775727, 4586532, 4399549, 4225369, 5148394, 2722564, 4160301, 4400556, 4560937,
    5175433, 5179511, 4788672, 3400041, 4438010, 4724818, 4909819, 4505816, 4506829, 5063571,
    4754048, 3359055, 4821766, 4681124, 4505859, 3727788, 4394651, 5684268, 4545473, 4807136,
    4682556, 4521089, 4788260, 4541024, 4505782, 4674297, 4307054, 4914525, 4681815, 4669026,
    4307000, 4306326, 4681800, 4160298, 4681866, 4681370, 4807139, 4750692, 4668893, 3727798,
    4284336, 3727799, 4307001, 5001276, 4762682, 4724824, 4585770, 4500524, 3727805, 5089619,
    5040568, 4505787, 4884587, 4788688, 4681807, 4394709, 5115114, 4807073, 4541016, 4553836,
    4505826, 5055171, 5031094, 4807031, 4505805, 4681368, 4435337, 3727807, 5031093, 4367595,
    4284338, 3400043, 5746213, 5115112, 5060586, 4770274, 4770259, 4682335, 4586559, 4160292,
    3727793, 4679572, 4668907, 4545470, 4536966, 4536918, 4365775, 4586083, 4541253, 4536965,
    4225370, 4160307, 3727808, 5156241, 4771193, 4734206, 4682328, 4681403, 4596821, 4586566,
    4550619, 4536002, 4505843, 4435256, 4416195, 4305771, 3727809, 3727801, 4738121, 4681725,
    4681681, 4586567, 4505797, 4395063, 4394643, 4365893, 4326135, 5010173, 4914400, 4807144,
    4738122, 4733517, 4681735, 4628497, 4534586, 4515879, 4395165, 4160297, 3727795, 5981717,
    5980565, 5746659, 5148573, 5148539, 5148421, 5089618, 4682554, 4668905, 4521055, 3400745,
    6094510, 5374918, 5336863, 5232897, 5063567, 5063549, 4586564, 4416194, 4395087, 4393527,
    4302422, 5746680, 5232895, 5031098, 4807142, 4770193, 4681886, 4679208, 4502495, 4416196,
]

print(f"📦 총 {len(TARGET_LIST)}개 상품 대상")

# ============================================================
# 실행
# ============================================================
state     = load_checkpoint()
remaining = [g for g in TARGET_LIST if state.get(str(g)) not in ("ok", "empty")]

if not remaining:
    print("✅ 모든 상품이 완료 상태입니다.")
else:
    session = build_session()
    api_cfg = diagnose_api(session, remaining[0])

    if not api_cfg:
        print("❌ API 진단 실패 — 네트워크/차단 여부 확인 필요")
    else:
        log.info(f"🚀 수집 시작 — [{api_cfg['name']}] / 남은 {len(remaining)}개")
        try:
            for g_no in tqdm(remaining, desc="상품"):
                try:
                    df, status = collect_product(session, g_no, api_cfg)
                except Exception as e:
                    log.exception(f"  ❌ 예외: {e}")
                    state[str(g_no)] = "fail"
                    save_checkpoint(state)
                    continue

                if status == "ok":
                    append_csv(df, OUTPUT_FILE)
                    log.info(f"  💾 {len(df)}건 저장 (상품 {g_no})")
                    state[str(g_no)] = "ok"
                elif status == "empty":
                    log.info(f"  📭 리뷰 없음 (상품 {g_no})")
                    state[str(g_no)] = "empty"
                else:
                    log.warning(f"  ⚠ 일시 실패 (상품 {g_no}) — 다음 실행 시 재시도")
                    state[str(g_no)] = "fail"

                save_checkpoint(state)
                time.sleep(SLEEP_BETWEEN_PROD)

        except KeyboardInterrupt:
            log.warning("⚠ 사용자 중단 — 체크포인트 저장 후 종료")
        finally:
            save_checkpoint(state)

        ok    = sum(1 for v in state.values() if v == "ok")
        empty = sum(1 for v in state.values() if v == "empty")
        fail  = sum(1 for v in state.values() if v == "fail")
        print(f"\n🎉 종료 — ok:{ok} / empty:{empty} / fail:{fail}")
        print(f"저장 파일: {OUTPUT_FILE}")

2026-04-21 23:09:37,022 [INFO] 🔍 API 진단 (상품 3297519)
2026-04-21 23:09:37,131 [INFO]   ✅ [type-style/p0] 총 리뷰 수: 18건 감지
2026-04-21 23:09:37,132 [INFO] 🚀 수집 시작 — [type-style/p0] / 남은 180개


📦 총 180개 상품 대상


상품:   0%|          | 0/180 [00:00<?, ?it/s]2026-04-21 23:09:37,201 [INFO]   ✔ 3297519/스타일/p0 — 18건
2026-04-21 23:09:38,050 [INFO]   3297519/스타일/p1 — 빈 페이지 (타입 종료)
2026-04-21 23:09:38,107 [INFO]   ✔ 3297519/일반/p0 — 20건
2026-04-21 23:09:38,991 [INFO]   ✔ 3297519/일반/p1 — 20건
2026-04-21 23:09:39,862 [INFO]   ✔ 3297519/일반/p2 — 20건
2026-04-21 23:09:40,743 [INFO]   ✔ 3297519/일반/p3 — 20건
2026-04-21 23:09:41,612 [INFO]   ✔ 3297519/일반/p4 — 20건
2026-04-21 23:09:42,502 [INFO]   ✔ 3297519/일반/p5 — 20건
2026-04-21 23:09:43,362 [INFO]   ✔ 3297519/일반/p6 — 20건
2026-04-21 23:09:44,479 [INFO]   ✔ 3297519/일반/p7 — 20건
2026-04-21 23:09:45,342 [INFO]   ✔ 3297519/일반/p8 — 20건
2026-04-21 23:09:46,276 [INFO]   ✔ 3297519/일반/p9 — 20건
2026-04-21 23:09:47,400 [INFO]   ✔ 3297519/일반/p10 — 20건
2026-04-21 23:09:48,261 [INFO]   ✔ 3297519/일반/p11 — 20건
2026-04-21 23:09:49,123 [INFO]   ✔ 3297519/일반/p12 — 20건
2026-04-21 23:09:50,002 [INFO]   ✔ 3297519/일반/p13 — 20건
2026-04-21 23:09:50,888 [INFO]   ✔ 3297519/일반/p14 — 20건
2026-04


🎉 종료 — ok:180 / empty:0 / fail:0
저장 파일: musinsa_andar_reviews.csv
